# Limpieza y Filtrado de Datos Energéticos

Procesamiento de `Base_energetica_formato_largo_2019_2024.xlsx` usando pandas y numpy.

In [1]:
import pandas as pd
import numpy as np

def limpiar_datos(ruta_archivo: str, hoja: str = 'Datos_Formato_Largo') -> pd.DataFrame:
    """
    Carga, limpia y estandariza el DataFrame de la base energética.
    
    Args:
        ruta_archivo: Ruta al archivo Excel.
        hoja: Nombre de la hoja a leer.
    
    Returns:
        DataFrame limpio con columnas renombradas.
    """
    # Cargar datos
    df = pd.read_excel(ruta_archivo, sheet_name=hoja)
    print(f'Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas')
    
    # 1. Eliminar duplicados
    df = df.drop_duplicates()
    print(f'Después de eliminar duplicados: {df.shape[0]} filas')
    
    # 2. Eliminar valores nulos en columnas clave
    columnas_clave = [
        'Año', 'Tipo de Energía', 'Fuente de Energía',
        'Categoría', 'Subcategoría', 'Valor (miles de TEP)'
    ]
    nulos_antes = df[columnas_clave].isnull().sum().sum()
    df = df.dropna(subset=columnas_clave)
    nulos_despues = df[columnas_clave].isnull().sum().sum()
    print(f'Valores nulos eliminados: {nulos_antes - nulos_despues}')
    print(f'Después de eliminar nulos: {df.shape[0]} filas')
    
    # 3. Crear copia y renombrar columnas
    df_clean = df.copy()
    mapa_columnas = {
        'Año': 'año',
        'Tipo de Energía': 'tipo_energia',
        'Fuente de Energía': 'fuente_energia',
        'Categoría': 'categoria',
        'Subcategoría': 'subcategoria',
        'Valor (miles de TEP)': 'valor_miles_TEP'
    }
    df_clean = df_clean.rename(columns=mapa_columnas)
    
    # 4. Optimizar tipos de datos (usando numpy)
    df_clean['año'] = df_clean['año'].astype(np.int16)
    df_clean['valor_miles_TEP'] = df_clean['valor_miles_TEP'].astype(np.float32)
    for col in ['tipo_energia', 'fuente_energia', 'categoria', 'subcategoria']:
        df_clean[col] = df_clean[col].astype('category')
    
    print(f'Limpieza completada. Shape final: {df_clean.shape}')
    print(f'Columnas: {df_clean.columns.tolist()}')
    
    return df_clean

In [2]:
ARCHIVO = 'Base_energetica_formato_largo_2019_2024.xlsx'

df_limpio = limpiar_datos(ARCHIVO)

Datos cargados: 4896 filas, 6 columnas
Después de eliminar duplicados: 4896 filas
Valores nulos eliminados: 240
Después de eliminar nulos: 4656 filas
Limpieza completada. Shape final: (4656, 6)
Columnas: ['año', 'tipo_energia', 'fuente_energia', 'categoria', 'subcategoria', 'valor_miles_TEP']


In [3]:
# Guardar resultado
df_limpio.to_csv('Base_energetica_limpia.csv', index=False)
print('Guardado en: Base_energetica_limpia.csv')

Guardado en: Base_energetica_limpia.csv


In [4]:
# Vista rápida de los datos
print('Primeras 5 filas:')
df_limpio.head()

Primeras 5 filas:


,año,tipo_energia,fuente_energia,categoria,subcategoria,valor_miles_TEP
0,2019,PRIMARIA,Aceites Vegetales,CENTROS DE TRANSFORMACIÓN,ACEITERAS Y DESTILERÍAS,-1992.359009
1,2019,PRIMARIA,Aceites Vegetales,CENTROS DE TRANSFORMACIÓN,ALTOS HORNOS,0.000000
2,2019,PRIMARIA,Aceites Vegetales,CENTROS DE TRANSFORMACIÓN,CARBONERAS,0.000000
3,2019,PRIMARIA,Aceites Vegetales,CENTROS DE TRANSFORMACIÓN,CENTRALES ELÉCTRICAS - AUTOPRODUCCIÓN,0.000000
4,2019,PRIMARIA,Aceites Vegetales,CENTROS DE TRANSFORMACIÓN,CENTRALES ELÉCTRICAS - SERVICIO PÚBLICO,0.000000


In [5]:
# Información del DataFrame
df_limpio.info()

<class 'pandas.DataFrame'>
Index: 4656 entries, 0 to 4855
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   año              4656 non-null   int16   
 1   tipo_energia     4656 non-null   category
 2   fuente_energia   4656 non-null   category
 3   categoria        4656 non-null   category
 4   subcategoria     4656 non-null   category
 5   valor_miles_TEP  4656 non-null   float32 
dtypes: category(4), float32(1), int16(1)
memory usage: 82.4 KB


In [6]:
# Estadísticas descriptivas
df_limpio.describe(include='all')

,año,tipo_energia,fuente_energia,categoria,subcategoria,valor_miles_TEP
count,4656.000000,4656,4656,4656,4656,4656.000000
unique,NaN,3,33,3,24,NaN
top,NaN,SECUNDARIA,Aceites Vegetales,CONSUMO,CONSUMO FINAL - AGROPECUARIO,NaN
freq,NaN,2736,144,1584,198,NaN
mean,2021.500000,NaN,NaN,NaN,NaN,1048.105713
std,1.708009,NaN,NaN,NaN,NaN,7628.428223
min,2019.000000,NaN,NaN,NaN,NaN,-40073.921875
25%,2020.000000,NaN,NaN,NaN,NaN,0.000000
50%,2021.500000,NaN,NaN,NaN,NaN,0.000000
75%,2023.000000,NaN,NaN,NaN,NaN,0.000000


In [7]:
# Valores únicos por columna categórica
for col in ['tipo_energia', 'fuente_energia', 'categoria', 'subcategoria']:
    print(f'{col}: {df_limpio[col].nunique()} valores únicos')
    print(df_limpio[col].unique())
    print()

tipo_energia: 3 valores únicos
['PRIMARIA', 'SECUNDARIA', 'TOTAL GENERAL']
Categories (3, str): ['PRIMARIA', 'SECUNDARIA', 'TOTAL GENERAL']

fuente_energia: 33 valores únicos
['Aceites Vegetales', 'Alcoholes Vegetales', 'Bagazo', 'Carbón Mineral', 'Energía Eólico', ..., 'Motonafta Total', 'No Energético', 'Otras Naftas', 'TOTAL II', 'ENERGÍA PRIMARIA + SECUNDARIA']
Length: 33
Categories (33, str): ['Aceites Vegetales', 'Alcoholes Vegetales', 'Bagazo', 'Biodiesel', ..., 'Otros Primarios', 'Petróleo', 'TOTAL I', 'TOTAL II']

categoria: 3 valores únicos
['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']
Categories (3, str): ['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']

subcategoria: 24 valores únicos
['ACEITERAS Y DESTILERÍAS', 'ALTOS HORNOS', 'CARBONERAS', 'CENTRALES ELÉCTRICAS - AUTOPRODUCCIÓN', 'CENTRALES ELÉCTRICAS - SERVICIO PÚBLICO', ..., 'NO APROVECHADO', 'OFERTA INTERNA', 'PRODUCCIÓN', 'PÉRDIDAS', 'VARIACIÓN DE STOCK']
Length: 24
Categories (24, str): ['ACEITERAS Y DESTILERÍ

# Filtrado de Energías

Procesamiento de los 3 grupos solicitados con cálculo de máximos y mínimos por categoría (CONSUMO, OFERTA) para años 2019-2024.

In [8]:
# Función auxiliar para calcular máximos y mínimos
def calcular_max_min(df_filtrado: pd.DataFrame, nombre_grupo: str) -> pd.DataFrame:
    """Calcula max y min por fuente_energia y categoria, con año y subcategoria."""
    resultados = []
    
    for fuente in df_filtrado['fuente_energia'].unique():
        for cat in ['CONSUMO', 'OFERTA']:
            sub_df = df_filtrado[
                (df_filtrado['fuente_energia'] == fuente) &
                (df_filtrado['categoria'] == cat)
            ]
            
            if len(sub_df) == 0:
                continue
            
            sub_df_valido = sub_df.dropna(subset=['valor_miles_TEP'])
            if len(sub_df_valido) == 0:
                continue
            
            idx_max = sub_df_valido['valor_miles_TEP'].idxmax()
            fila_max = sub_df_valido.loc[idx_max]
            
            idx_min = sub_df_valido['valor_miles_TEP'].idxmin()
            fila_min = sub_df_valido.loc[idx_min]
            
            resultados.append({
                'grupo': nombre_grupo,
                'fuente_energia': fuente,
                'categoria': cat,
                'max_valor': fila_max['valor_miles_TEP'],
                'max_año': int(fila_max['año']),
                'max_subcategoria': fila_max['subcategoria'],
                'min_valor': fila_min['valor_miles_TEP'],
                'min_año': int(fila_min['año']),
                'min_subcategoria': fila_min['subcategoria']
            })
    
    return pd.DataFrame(resultados)

## 1. Energías Primarias: Petróleo, Hidráulica, Nuclear, Gas

In [9]:
# Filtrado energías primarias
fuentes_primarias = [
    'Petróleo',
    'Energía Hidráulica',
    'Energía Nuclear',
    'Gas Natural de Pozo'
]

energias_primaria = df_limpio[
    (df_limpio['tipo_energia'] == 'PRIMARIA') &
    (df_limpio['fuente_energia'].isin(fuentes_primarias))
].copy()

print(f'Filas: {energias_primaria.shape[0]}')
print(f'Fuentes: {energias_primaria["fuente_energia"].unique()}')
print(f'Categorías: {energias_primaria["categoria"].unique()}')
print(f'Años: {sorted(energias_primaria["año"].unique())}')

# Verificación: conteo por fuente y categoría
print('\nConteo por fuente y categoría:')
print(energias_primaria.groupby(['fuente_energia', 'categoria']).size().unstack(fill_value=0))

Filas: 576
Fuentes: ['Energía Hidráulica', 'Energía Nuclear', 'Gas Natural de Pozo', 'Petróleo']
Categories (33, str): ['Aceites Vegetales', 'Alcoholes Vegetales', 'Bagazo', 'Biodiesel', ..., 'Otros Primarios', 'Petróleo', 'TOTAL I', 'TOTAL II']
Categorías: ['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']
Categories (3, str): ['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']
Años: [np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024)]

Conteo por fuente y categoría:
categoria            CENTROS DE TRANSFORMACIÓN  CONSUMO  OFERTA
fuente_energia                                                 
Energía Hidráulica                          48       48      48
Energía Nuclear                             48       48      48
Gas Natural de Pozo                         48       48      48
Petróleo                                    48       48      48


In [10]:
# Máximos y mínimos - Energías Primarias
primaria_maxmin = calcular_max_min(energias_primaria, 'PRIMARIA')
print('Máximos y mínimos - Energías Primarias:')
display(primaria_maxmin)

Máximos y mínimos - Energías Primarias:


,grupo,fuente_energia,categoria,max_valor,max_año,max_subcategoria,min_valor,min_año,min_subcategoria
0,PRIMARIA,Energía Hidráulica,CONSUMO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO
1,PRIMARIA,Energía Hidráulica,OFERTA,3508.178467,2023,PRODUCCIÓN,-34.734440,2023,PÉRDIDAS
2,PRIMARIA,Energía Nuclear,CONSUMO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO
3,PRIMARIA,Energía Nuclear,OFERTA,2900.076660,2024,IMPORTACIÓN,0.000000,2019,AJUSTES
4,PRIMARIA,Gas Natural de Pozo,CONSUMO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO,-4356.802734,2019,CONSUMO PROPIO
5,PRIMARIA,Gas Natural de Pozo,OFERTA,44412.421875,2024,PRODUCCIÓN,-1603.692017,2024,PÉRDIDAS
6,PRIMARIA,Petróleo,CONSUMO,0.486566,2024,CONSUMO FINAL - TOTAL,-118.777367,2024,CONSUMO PROPIO
7,PRIMARIA,Petróleo,OFERTA,36292.750000,2024,PRODUCCIÓN,-9656.600586,2024,EXPORTACIÓN Y BUNKER


## 2. Energías Secundarias: Eléctrica, Gas de redes, Naftas

In [11]:
# Filtrado energías secundarias
fuentes_secundarias = [
    'Energía Eléctrica',
    'Gas Distribuido por Redes',
    'Gas Licuado',
    'Motonafta Total',
    'Biodiesel'
]

energia_secundaria = df_limpio[
    (df_limpio['tipo_energia'] == 'SECUNDARIA') &
    (df_limpio['fuente_energia'].isin(fuentes_secundarias))
].copy()

print(f'Filas: {energia_secundaria.shape[0]}')
print(f'Fuentes: {energia_secundaria["fuente_energia"].unique()}')
print(f'Categorías: {energia_secundaria["categoria"].unique()}')
print(f'Años: {sorted(energia_secundaria["año"].unique())}')

# Verificación: conteo por fuente y categoría
print('\nConteo por fuente y categoría:')
print(energia_secundaria.groupby(['fuente_energia', 'categoria']).size().unstack(fill_value=0))

Filas: 720
Fuentes: ['Biodiesel', 'Energía Eléctrica', 'Gas Distribuido por Redes', 'Gas Licuado', 'Motonafta Total']
Categories (33, str): ['Aceites Vegetales', 'Alcoholes Vegetales', 'Bagazo', 'Biodiesel', ..., 'Otros Primarios', 'Petróleo', 'TOTAL I', 'TOTAL II']
Categorías: ['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']
Categories (3, str): ['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']
Años: [np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024)]

Conteo por fuente y categoría:
categoria                  CENTROS DE TRANSFORMACIÓN  CONSUMO  OFERTA
fuente_energia                                                       
Biodiesel                                         48       48      48
Energía Eléctrica                                 48       48      48
Gas Distribuido por Redes                         48       48      48
Gas Licuado                                       48       48      48
Motonafta Total                            

In [12]:
# Máximos y mínimos - Energías Secundarias
secundaria_maxmin = calcular_max_min(energia_secundaria, 'SECUNDARIA')
print('Máximos y mínimos - Energías Secundarias:')
display(secundaria_maxmin)

Máximos y mínimos - Energías Secundarias:


,grupo,fuente_energia,categoria,max_valor,max_año,max_subcategoria,min_valor,min_año,min_subcategoria
0,SECUNDARIA,Biodiesel,CONSUMO,1011.714661,2019,CONSUMO FINAL - TOTAL,0.000000,2019,CONSUMO FINAL - AGROPECUARIO
1,SECUNDARIA,Biodiesel,OFERTA,1911.070679,2019,PRODUCCIÓN,-1129.916382,2021,EXPORTACIÓN Y BUNKER
2,SECUNDARIA,Energía Eléctrica,CONSUMO,11099.626953,2021,CONSUMO FINAL - TOTAL,-366.428589,2021,CONSUMO PROPIO
3,SECUNDARIA,Energía Eléctrica,OFERTA,13177.712891,2024,PRODUCCIÓN,-3047.532471,2023,PÉRDIDAS
4,SECUNDARIA,Gas Distribuido por Redes,CONSUMO,19896.876953,2022,CONSUMO FINAL - TOTAL,-773.559998,2019,CONSUMO PROPIO
5,SECUNDARIA,Gas Distribuido por Redes,OFERTA,35768.750000,2024,PRODUCCIÓN,-3288.459961,2020,PÉRDIDAS
6,SECUNDARIA,Gas Licuado,CONSUMO,1994.300049,2023,CONSUMO FINAL - TOTAL,-82.500000,2021,CONSUMO PROPIO
7,SECUNDARIA,Gas Licuado,OFERTA,3311.000000,2023,PRODUCCIÓN,-1394.800049,2024,EXPORTACIÓN Y BUNKER
8,SECUNDARIA,Motonafta Total,CONSUMO,6805.377930,2023,CONSUMO FINAL - TOTAL,-0.012770,2019,CONSUMO PROPIO
9,SECUNDARIA,Motonafta Total,OFERTA,6805.377930,2023,OFERTA INTERNA,-120.342995,2020,EXPORTACIÓN Y BUNKER


## 3. Energías Renovables: Eólica, Solar, Alcoholes Vegetales, Aceites Vegetales

In [13]:
# Filtrado energías renovables (incluye Eólica aunque no tenga datos)
fuentes_renovables = [
    'Energía Eólica',
    'Energía Solar',
    'Alcoholes Vegetales',
    'Aceites Vegetales'
]

# Ajustar nombres a los reales en los datos
fuentes_renovables_ajustadas = []
for f in fuentes_renovables:
    matches = df_limpio[df_limpio['fuente_energia'].str.contains(f.replace('ó', 'o').replace('í', 'i'), case=False, na=False)]['fuente_energia'].unique()
    if len(matches) > 0:
        fuentes_renovables_ajustadas.append(matches[0])
    else:
        fuentes_renovables_ajustadas.append(f)

print(f'Fuentes ajustadas: {fuentes_renovables_ajustadas}')

energia_renovable = df_limpio[
    df_limpio['fuente_energia'].isin(fuentes_renovables_ajustadas)
].copy()

# Añadir filas vacías para fuentes sin datos (ej. Eólica)
for fuente in fuentes_renovables_ajustadas:
    if fuente not in energia_renovable['fuente_energia'].values:
        for cat in ['CONSUMO', 'OFERTA', 'CENTROS DE TRANSFORMACIÓN']:
            for año in range(2019, 2025):
                fila_vacia = pd.DataFrame([{
                    'año': año,
                    'tipo_energia': 'PRIMARIA',
                    'fuente_energia': fuente,
                    'categoria': cat,
                    'subcategoria': 'SIN DATOS',
                    'valor_miles_TEP': np.nan
                }])
                energia_renovable = pd.concat([energia_renovable, fila_vacia], ignore_index=True)

print(f'Filas: {energia_renovable.shape[0]}')
print(f'Fuentes: {energia_renovable["fuente_energia"].unique()}')
print(f'Tipo energía: {energia_renovable["tipo_energia"].unique()}')
print(f'Categorías: {energia_renovable["categoria"].unique()}')
print(f'Años: {sorted(energia_renovable["año"].unique())}')

# Verificación: conteo por fuente y categoría
print('\nConteo por fuente y categoría:')
print(energia_renovable.groupby(['fuente_energia', 'categoria']).size().unstack(fill_value=0))

Fuentes ajustadas: ['Energía Eólica', 'Energía Solar', 'Alcoholes Vegetales', 'Aceites Vegetales']


Filas: 450
Fuentes: <StringArray>
['Aceites Vegetales', 'Alcoholes Vegetales', 'Energía Solar',
 'Energía Eólica']
Length: 4, dtype: str
Tipo energía: <StringArray>
['PRIMARIA']
Length: 1, dtype: str
Categorías: <StringArray>
['CENTROS DE TRANSFORMACIÓN', 'CONSUMO', 'OFERTA']
Length: 3, dtype: str
Años: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Conteo por fuente y categoría:
categoria            CENTROS DE TRANSFORMACIÓN  CONSUMO  OFERTA
fuente_energia                                                 
Aceites Vegetales                           48       48      48
Alcoholes Vegetales                         48       48      48
Energía Eólica                               6        6       6
Energía Solar                               48       48      48


In [14]:
# Máximos y mínimos - Energías Renovables
renovable_maxmin = calcular_max_min(energia_renovable, 'RENOVABLE')
print('Máximos y mínimos - Energías Renovables:')
display(renovable_maxmin)

Máximos y mínimos - Energías Renovables:


,grupo,fuente_energia,categoria,max_valor,max_año,max_subcategoria,min_valor,min_año,min_subcategoria
0,RENOVABLE,Aceites Vegetales,CONSUMO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO,0.0,2019,CONSUMO FINAL - AGROPECUARIO
1,RENOVABLE,Aceites Vegetales,OFERTA,1992.359009,2019,OFERTA INTERNA,0.0,2019,AJUSTES
2,RENOVABLE,Alcoholes Vegetales,CONSUMO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO,0.0,2019,CONSUMO FINAL - AGROPECUARIO
3,RENOVABLE,Alcoholes Vegetales,OFERTA,630.013367,2024,OFERTA INTERNA,0.0,2019,AJUSTES
4,RENOVABLE,Energía Solar,CONSUMO,0.000000,2019,CONSUMO FINAL - AGROPECUARIO,0.0,2019,CONSUMO FINAL - AGROPECUARIO
5,RENOVABLE,Energía Solar,OFERTA,342.240906,2024,OFERTA INTERNA,0.0,2019,AJUSTES


## Resumen de Verificación

In [15]:
print('=== RESUMEN DE VARIABLES CREADAS ===')
print(f'energias_primaria:      {energias_primaria.shape[0]} filas, {energias_primaria["fuente_energia"].nunique()} fuentes')
print(f'energia_secundaria:    {energia_secundaria.shape[0]} filas, {energia_secundaria["fuente_energia"].nunique()} fuentes')
print(f'energia_renovable:     {energia_renovable.shape[0]} filas, {energia_renovable["fuente_energia"].nunique()} fuentes')
print()
print('=== DATAFRAMES MAX/MIN ===')
print(f'primaria_maxmin:   {primaria_maxmin.shape[0]} filas')
print(f'secundaria_maxmin: {secundaria_maxmin.shape[0]} filas')
print(f'renovable_maxmin:  {renovable_maxmin.shape[0]} filas')
print()
print('✅ Todas las variables cargadas en memoria (no se guardaron CSV)')
print()
print('Variables disponibles:')
print('  - energias_primaria, primaria_maxmin')
print('  - energia_secundaria, secundaria_maxmin')
print('  - energia_renovable, renovable_maxmin')

=== RESUMEN DE VARIABLES CREADAS ===
energias_primaria:      576 filas, 4 fuentes
energia_secundaria:    720 filas, 5 fuentes
energia_renovable:     450 filas, 4 fuentes

=== DATAFRAMES MAX/MIN ===
primaria_maxmin:   8 filas
secundaria_maxmin: 10 filas
renovable_maxmin:  6 filas

✅ Todas las variables cargadas en memoria (no se guardaron CSV)

Variables disponibles:
  - energias_primaria, primaria_maxmin
  - energia_secundaria, secundaria_maxmin
  - energia_renovable, renovable_maxmin


## Guardar en Excel (múltiples hojas)

In [16]:
# Guardar todos los DataFrames en un solo archivo Excel con múltiples hojas
archivo_salida = 'Filtros_Energeticos_Analisis.xlsx'

with pd.ExcelWriter(archivo_salida, engine='openpyxl') as writer:
    # Energías Primarias
    energias_primaria.to_excel(writer, sheet_name='Energia_Primarias', index=False)
    primaria_maxmin.to_excel(writer, sheet_name='Max_Min_Primarias', index=False)
    
    # Energías Secundarias
    energia_secundaria.to_excel(writer, sheet_name='Energia_Secundarias', index=False)
    secundaria_maxmin.to_excel(writer, sheet_name='Max_Min_Secundarias', index=False)
    
    # Energías Renovables
    energia_renovable.to_excel(writer, sheet_name='Energias_Renovables', index=False)
    renovable_maxmin.to_excel(writer, sheet_name='Max_Min_Renovables', index=False)

print(f'✅ Archivo guardado: {archivo_salida}')
print('Hojas creadas:')
print('  - Energia_Primarias')
print('  - Max_Min_Primarias')
print('  - Energia_Secundarias')
print('  - Max_Min_Secundarias')
print('  - Energias_Renovables')
print('  - Max_Min_Renovables')

✅ Archivo guardado: Filtros_Energeticos_Analisis.xlsx
Hojas creadas:
  - Energia_Primarias
  - Max_Min_Primarias
  - Energia_Secundarias
  - Max_Min_Secundarias
  - Energias_Renovables
  - Max_Min_Renovables


In [17]:
# Guardar todos los DataFrames en un solo archivo Excel con multiples hojas
archivo_salida = "Filtros_Energeticos_Analisis.xlsx"

with pd.ExcelWriter(archivo_salida, engine="openpyxl") as writer:
    # Energia Primarias
    energias_primaria.to_excel(writer, sheet_name="Energia_Primarias", index=False)
    primaria_maxmin.to_excel(writer, sheet_name="Max_Min_Primarias", index=False)
    
    # Energia Secundarias
    energia_secundaria.to_excel(writer, sheet_name="Energia_Secundarias", index=False)
    secundaria_maxmin.to_excel(writer, sheet_name="Max_Min_Secundarias", index=False)
    
    # Energias Renovables
    energia_renovable.to_excel(writer, sheet_name="Energias_Renovables", index=False)
    renovable_maxmin.to_excel(writer, sheet_name="Max_Min_Renovables", index=False)

print(f"Archivo guardado: {archivo_salida}")
print("Hojas creadas:")
print("  - Energia_Primarias / Max_Min_Primarias")
print("  - Energia_Secundarias / Max_Min_Secundarias")
print("  - Energias_Renovables / Max_Min_Renovables")

Archivo guardado: Filtros_Energeticos_Analisis.xlsx
Hojas creadas:
  - Energia_Primarias / Max_Min_Primarias
  - Energia_Secundarias / Max_Min_Secundarias
  - Energias_Renovables / Max_Min_Renovables


In [ ]:
# Guardar todos los DataFrames en un solo archivo Excel con multiples hojas
archivo_salida = "Filtros_Energeticos_Analisis.xlsx"

with pd.ExcelWriter(archivo_salida, engine="openpyxl") as writer:
    # Energias Primarias
    energias_primaria.to_excel(writer, sheet_name="Energia_Primarias", index=False)
    primaria_maxmin.to_excel(writer, sheet_name="Max_Min", index=False)
    
    # Energia Secundarias
    energia_secundaria.to_excel(writer, sheet_name="Energia_Secundarias", index=False)
    secundaria_maxmin.to_excel(writer, sheet_name="Max_Min", index=False)
    
    # Energias Renovables
    energia_renovable.to_excel(writer, sheet_name="Energias_Renovables", index=False)
    renovable_maxmin.to_excel(writer, sheet_name="Max_Min", index=False)

print(f"Archivo guardado: {archivo_salida}")
print("Hojas creadas:")
print("  - Energia_Primarias / Max_Min")
print("  - Energia_Secundarias / Max_Min")
print("  - Energias_Renovables / Max_Min")